# NB01 — Data Preparation
**Project:** CNN vs ViT Localization Faithfulness in Chest X-Ray  
**Stage:** 1 — Run after NB00 is complete  

## What this notebook does
1. Mounts Drive and installs packages  
2. Resizes 1024×1024 PNGs → 224×224 (model input size)  
3. Parses annotations and builds radiologist consensus boxes  
4. Computes Fleiss' κ (inter-rater reliability per pathology)  
5. Creates multi-label stratified train/val/test splits  
6. Saves all outputs to Drive  

## Before running
- NB00 must be complete ✅  
- `train_images/` must have 15,000 PNGs ✅  
- `annotations/train.csv` must exist ✅  
- Runtime does NOT need GPU (CPU is fine)

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_ROOT = '/content/drive/MyDrive/cxr_faithfulness'
print("✅ Drive mounted:", GDRIVE_ROOT)

Mounted at /content/drive
✅ Drive mounted: /content/drive/MyDrive/cxr_faithfulness


In [ ]:
# Auto-install missing packages
import sys
exec(open(f'{GDRIVE_ROOT}/config/startup.py').read())

⏳ Installing missing packages (~1 min)...
✅ Packages ready. Ignore any conflict warnings.


In [ ]:
# Load config and set all paths
import sys
sys.path.insert(0, f'{GDRIVE_ROOT}/config')

from pathlib import Path
import pandas as pd
import numpy as np
import cv2
import os
import shutil

KAGGLE_DIR       = Path(GDRIVE_ROOT) / "data" / "raw" / "kaggle"
TRAIN_IMAGES_RAW = KAGGLE_DIR / "train_images"
ANNOTATIONS_PATH = KAGGLE_DIR / "annotations"

PROCESSED_PATH   = Path(GDRIVE_ROOT) / "data" / "processed"
IMAGES_OUT       = PROCESSED_PATH / "images"
SPLITS_OUT       = PROCESSED_PATH / "splits"
CONSENSUS_OUT    = PROCESSED_PATH / "consensus"

for p in [IMAGES_OUT, SPLITS_OUT, CONSENSUS_OUT]:
    p.mkdir(parents=True, exist_ok=True)

IMG_SIZE       = 224
CONSENSUS_RULE = 2   # >=2 of 3 radiologists must agree
N_READERS      = 3

print("✅ Paths configured.")
print(f"   Input images  : {TRAIN_IMAGES_RAW}")
print(f"   Output images : {IMAGES_OUT}")
print(f"   Splits output : {SPLITS_OUT}")

✅ Paths configured.
   Input images  : /content/drive/MyDrive/cxr_faithfulness/data/raw/kaggle/train_images
   Output images : /content/drive/MyDrive/cxr_faithfulness/data/processed/images
   Splits output : /content/drive/MyDrive/cxr_faithfulness/data/processed/splits


## Step 1 — Resize images to 224×224
Each 1024×1024 PNG is resized using `cv2.INTER_AREA` (best quality for downscaling).  
Augmentation is NOT done here — it happens inside the DataLoader during training only.  
Existence check: already-processed images are skipped so this cell is safe to re-run.

In [ ]:
all_pngs   = sorted(TRAIN_IMAGES_RAW.glob("*.png"))
total      = len(all_pngs)
converted  = 0
skipped    = 0
errors     = []

print(f"Found {total} images to process...")

for i, png_path in enumerate(all_pngs):
    dest = IMAGES_OUT / png_path.name
    if dest.exists():
        skipped += 1
        continue

    try:
        img = cv2.imread(str(png_path))
        if img is None:
            raise ValueError(f"cv2 could not read: {png_path.name}")

        img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(dest), img_resized)
        converted += 1

    except Exception as e:
        errors.append(str(png_path.name))

    if (i + 1) % 1000 == 0:
        print(f"   Processed {i+1}/{total}...")

# Save error log
error_log = PROCESSED_PATH / "conversion_errors.txt"
with open(str(error_log), "w") as f:
    f.write("\n".join(errors))

print()
print(f"✅ Converted  : {converted}")
print(f"⏭️  Skipped    : {skipped} (already existed)")
print(f"❌ Errors     : {len(errors)}")
if errors:
    print(f"   Error log  : {error_log}")

Found 15000 images to process...
   Processed 1000/15000...
   Processed 2000/15000...
   Processed 3000/15000...
   Processed 4000/15000...
   Processed 5000/15000...
   Processed 6000/15000...
   Processed 7000/15000...
   Processed 8000/15000...
   Processed 9000/15000...
   Processed 10000/15000...
   Processed 11000/15000...
   Processed 12000/15000...
   Processed 13000/15000...
   Processed 14000/15000...
   Processed 15000/15000...

✅ Converted  : 15000
⏭️  Skipped    : 0 (already existed)
❌ Errors     : 0


## Step 2 — Load and inspect annotations
Load `train.csv` and confirm it has the expected structure.

In [ ]:
train_df = pd.read_csv(str(ANNOTATIONS_PATH / "train.csv"))

print(f"Total rows        : {len(train_df)}")
print(f"Columns           : {list(train_df.columns)}")
print(f"Unique image IDs  : {train_df['image_id'].nunique()}")
print(f"Unique classes    : {sorted(train_df['class_name'].unique().tolist())}")
print(f"Radiologist IDs   : {sorted(train_df['rad_id'].unique().tolist())}")
print()
display(train_df.head())

Total rows        : 67914
Columns           : ['image_id', 'class_name', 'class_id', 'rad_id', 'x_min', 'y_min', 'x_max', 'y_max', 'width', 'height']
Unique image IDs  : 15000
Unique classes    : ['Aortic enlargement', 'Atelectasis', 'Calcification', 'Cardiomegaly', 'Consolidation', 'ILD', 'Infiltration', 'Lung Opacity', 'No finding', 'Nodule/Mass', 'Other lesion', 'Pleural effusion', 'Pleural thickening', 'Pneumothorax', 'Pulmonary fibrosis']
Radiologist IDs   : ['R1', 'R10', 'R11', 'R12', 'R13', 'R14', 'R15', 'R16', 'R17', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8', 'R9']



,image_id,class_name,class_id,rad_id,x_min,y_min,x_max,y_max,width,height
0,50a418190bc3fb1ef1633bf9678929b3,No finding,14,R11,NaN,NaN,NaN,NaN,2332,2580
1,21a10246a5ec7af151081d0cd6d65dc9,No finding,14,R7,NaN,NaN,NaN,NaN,2954,3159
2,9a5094b2563a1ef3ff50dc5c7ff71345,Cardiomegaly,3,R10,691.0,1375.0,1653.0,1831.0,2080,2336
3,051132a778e61a86eb147c7c6f564dfe,Aortic enlargement,0,R10,1264.0,743.0,1611.0,1019.0,2304,2880
4,063319de25ce7edb9b1c6b8881290140,No finding,14,R10,NaN,NaN,NaN,NaN,2540,3072


In [ ]:
l =list(train_df['class_name'].unique())
print(l)
print(len(l))

['No finding', 'Cardiomegaly', 'Aortic enlargement', 'Pleural thickening', 'ILD', 'Nodule/Mass', 'Pulmonary fibrosis', 'Lung Opacity', 'Atelectasis', 'Other lesion', 'Infiltration', 'Pleural effusion', 'Calcification', 'Consolidation', 'Pneumothorax']
15


## Step 3 — Build radiologist consensus boxes
For each image × pathology combination:  
- Count how many radiologists (of 3) annotated that finding  
- Keep only findings agreed by `CONSENSUS_RULE` (≥2 of 3)  
- Merge bounding boxes via **union coordinates** (largest box covering all annotations)  
- Save both 2-of-3 and 3-of-3 consensus files for sensitivity analysis

In [ ]:
# Exclude "No finding" rows from consensus box building
finding_df = train_df[train_df['class_name'] != 'No finding'].copy()
no_finding_df = train_df[train_df['class_name'] == 'No finding'].copy()

# 1. Load image dimensions from train_meta.csv
meta_df = pd.read_csv(str(CONSENSUS_OUT / "train_meta.csv"))

# 2. Merge dimensions into finding_df
finding_df = finding_df.merge(meta_df[['image_id', 'dim0', 'dim1']], on='image_id', how='left')

# 3. Scale bounding boxes to IMG_SIZE (224)
# dim1 is original width (x), dim0 is original height (y)
finding_df['scale_x'] = IMG_SIZE / finding_df['dim1']
finding_df['scale_y'] = IMG_SIZE / finding_df['dim0']

finding_df['x_min'] = finding_df['x_min'] * finding_df['scale_x']
finding_df['y_min'] = finding_df['y_min'] * finding_df['scale_y']
finding_df['x_max'] = finding_df['x_max'] * finding_df['scale_x']
finding_df['y_max'] = finding_df['y_max'] * finding_df['scale_y']

print(f"Finding rows     : {len(finding_df)}")
print(f"No finding rows  : {len(no_finding_df)}")

def build_consensus(df, rule):
    """Build consensus boxes for a given minimum reader agreement rule."""
    results = []
    grouped = df.groupby(['image_id', 'class_name'])

    for (image_id, class_name), group in grouped:
        n_readers = group['rad_id'].nunique()
        if n_readers >= rule:
            # Union box — take min/max across all reader boxes
            x_min = group['x_min'].min()
            y_min = group['y_min'].min()
            x_max = group['x_max'].max()
            y_max = group['y_max'].max()
            results.append({
                'image_id'   : image_id,
                'class_name' : class_name,
                'n_readers'  : n_readers,
                'x_min'      : round(x_min, 4),
                'y_min'      : round(y_min, 4),
                'x_max'      : round(x_max, 4),
                'y_max'      : round(y_max, 4),
            })

    return pd.DataFrame(results)

# 4. Build 1-of-3, 2-of-3, and 3-of-3 consensus
consensus_1of3 = build_consensus(finding_df, rule=1)
consensus_2of3 = build_consensus(finding_df, rule=2)
consensus_3of3 = build_consensus(finding_df, rule=3)

# 5. Save all three files
consensus_1of3.to_csv(str(CONSENSUS_OUT / "consensus_boxes_1of3.csv"), index=False)
consensus_2of3.to_csv(str(CONSENSUS_OUT / "consensus_boxes_2of3.csv"), index=False)
consensus_3of3.to_csv(str(CONSENSUS_OUT / "consensus_boxes_3of3.csv"), index=False)

print(f"✅ 1-of-3 consensus boxes : {len(consensus_1of3)} (saved)")
print(f"✅ 2-of-3 consensus boxes : {len(consensus_2of3)} (saved)")
print(f"✅ 3-of-3 consensus boxes : {len(consensus_3of3)} (saved)")
display(consensus_2of3.head())

Finding rows     : 36096
No finding rows  : 31818
✅ 1-of-3 consensus boxes : 15365 (saved)
✅ 2-of-3 consensus boxes : 8825 (saved)
✅ 3-of-3 consensus boxes : 5266 (saved)


,image_id,class_name,n_readers,x_min,y_min,x_max,y_max
0,0005e8e3701dfb1dd93d53e2ff537b6e,Lung Opacity,2,65.6250,42.5104,87.8646,64.8958
1,0007d316f756b3fa0baea2ff514ce945,Aortic enlargement,2,120.0694,79.4111,145.6389,99.6333
2,0007d316f756b3fa0baea2ff514ce945,Pleural thickening,3,60.3750,49.3111,101.7917,73.0333
3,0007d316f756b3fa0baea2ff514ce945,Pulmonary fibrosis,2,78.1667,52.6556,147.5833,73.0333
4,000d68e42b71d3eac10ccc077aba07c1,Lung Opacity,2,16.5278,19.1333,197.7500,115.2667


## Step 4 — Fleiss' κ (inter-rater reliability)
Measures how consistently the 3 radiologists agree on each pathology.  
κ > 0.6 = substantial agreement, κ > 0.8 = almost perfect.  
This becomes the κ column in Table 1 and Table 2 of the paper.

In [ ]:
from statsmodels.stats.inter_rater import fleiss_kappa

kappa_results = []

all_classes = [c for c in train_df['class_name'].unique() if c != 'No finding']
all_image_ids = train_df['image_id'].unique()

# Data quality check: ensure every image has EXACTLY N_READERS raters
actual_readers = train_df.groupby('image_id')['rad_id'].nunique()
assert (actual_readers == N_READERS).all(), \
    f"Images with unexpected rater count: " \
    f"{actual_readers[actual_readers != N_READERS].to_dict()}"

for cls in sorted(all_classes):
    cls_df = train_df[train_df['class_name'] == cls]

    # 1. Count how many raters annotated this pathology per image
    pos_counts = cls_df.groupby('image_id')['rad_id'].nunique()

    # 2. Reindex to include all images (fill missing with 0 positive ratings)
    pos_counts = pos_counts.reindex(all_image_ids, fill_value=0)

    # 3. Build the exact aggregation matrix Fleiss' Kappa expects
    # (Rows = images, Col 0 = Negative ratings, Col 1 = Positive ratings)
    pos_array = pos_counts.values
    neg_array = N_READERS - pos_array

    # Defensive clipping just in case of annotation anomalies
    pos_array = np.clip(pos_array, 0, N_READERS)
    neg_array = np.clip(neg_array, 0, N_READERS)

    agg = np.column_stack((neg_array, pos_array))

    try:
        kappa = fleiss_kappa(agg)
    except Exception as e:
        print(f"  ⚠️  Kappa failed for '{cls}': {e}")
        kappa = None

    kappa_results.append({
        'class_name' : cls,
        'fleiss_kappa': round(kappa, 4) if kappa is not None else None,
        'n_images_any_positive': int((pos_array >= 1).sum()),
        'n_images_consensus_positive': int((pos_array >= 2).sum())
    })

def interpret_kappa(k):
    if k is None:     return "N/A"
    elif k < 0.0:     return "Poor"
    elif k < 0.2:     return "Slight"
    elif k < 0.4:     return "Fair"
    elif k < 0.6:     return "Moderate"
    elif k < 0.8:     return "Substantial"
    else:             return "Almost perfect"

kappa_df = pd.DataFrame(kappa_results).sort_values('fleiss_kappa', ascending=False)
kappa_df['interpretation'] = kappa_df['fleiss_kappa'].apply(interpret_kappa)
kappa_df.to_csv(str(PROCESSED_PATH / "fleiss_kappa.csv"), index=False)

print("✅ Fleiss' κ computed and saved.")
display(kappa_df)

✅ Fleiss' κ computed and saved.


,class_name,fleiss_kappa,n_images_any_positive,n_images_consensus_positive,interpretation
3,Cardiomegaly,0.7894,2300,1817,Substantial
0,Aortic enlargement,0.7802,3067,2346,Substantial
12,Pneumothorax,0.7426,96,58,Substantial
10,Pleural effusion,0.7045,1032,634,Substantial
13,Pulmonary fibrosis,0.6752,1617,1017,Substantial
8,Nodule/Mass,0.5284,826,405,Moderate
5,ILD,0.4757,386,152,Moderate
11,Pleural thickening,0.4607,1981,882,Moderate
6,Infiltration,0.4381,613,245,Moderate
2,Calcification,0.4153,452,177,Moderate


## Step 5 — Build image-level multi-label dataframe
Each image gets one row with a binary vector of 14 pathology classes.  
This is needed for multi-label stratified splitting.

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f"✅ Seed set: {seed}")

RANDOM_SEED = 42
set_seed(RANDOM_SEED)

✅ Seed set: 42


In [ ]:
all_classes = sorted([c for c in train_df['class_name'].unique() if c != 'No finding'])
all_image_ids = train_df['image_id'].unique().tolist()

print(f"Total image IDs in CSV     : {len(all_image_ids)}")
print(f"Images found in processed/ : {len([i for i in all_image_ids if (IMAGES_OUT / f'{i}.png').exists()])}")

# Build multi-label binary matrix from pathology-only rows
label_pivot = train_df[train_df['class_name'] != 'No finding'].pivot_table(
    index='image_id', columns='class_name', aggfunc='size', fill_value=0
).clip(upper=1).reset_index()

# ✅ KEY FIX: merge back ALL image IDs so No finding-only images are included
all_ids_df = pd.DataFrame({'image_id': all_image_ids})
label_pivot = all_ids_df.merge(label_pivot, on='image_id', how='left').fillna(0)

# Ensure all 14 pathology columns exist and are int
for cls in all_classes:
    if cls not in label_pivot.columns:
        label_pivot[cls] = 0
    label_pivot[cls] = label_pivot[cls].astype(int)

# Add no_finding column
nf_ids = set(train_df[train_df['class_name'] == 'No finding']['image_id'])
label_pivot['no_finding'] = label_pivot['image_id'].isin(nf_ids).astype(int)

# Filter to valid image files
valid_ids = {img_id for img_id in label_pivot['image_id']
             if (IMAGES_OUT / f"{img_id}.png").exists()}
label_df = label_pivot[label_pivot['image_id'].isin(valid_ids)].reset_index(drop=True)

print(f"✅ Multi-label dataframe: {label_df.shape}")
display(label_df.head())

Total image IDs in CSV     : 15000
Images found in processed/ : 15000
✅ Multi-label dataframe: (15000, 16)


,image_id,Aortic enlargement,Atelectasis,Calcification,Cardiomegaly,Consolidation,ILD,Infiltration,Lung Opacity,Nodule/Mass,Other lesion,Pleural effusion,Pleural thickening,Pneumothorax,Pulmonary fibrosis,no_finding
0,50a418190bc3fb1ef1633bf9678929b3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,21a10246a5ec7af151081d0cd6d65dc9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,9a5094b2563a1ef3ff50dc5c7ff71345,1,0,0,1,0,0,0,0,0,0,1,1,0,0,0
3,051132a778e61a86eb147c7c6f564dfe,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0
4,063319de25ce7edb9b1c6b8881290140,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


## Step 6 — Multi-label stratified train/val/test split
80% train / 10% val / 10% test  
Uses `iterative_train_test_split` from scikit-multilearn — the only correct method for multi-label stratification.  
⚠️ Do NOT use sklearn's StratifiedShuffleSplit — it ignores multi-label structure.

In [ ]:
from skmultilearn.model_selection import iterative_train_test_split

label_cols = all_classes
X = label_df[['image_id']].values
y = label_df[label_cols].values

# First split: 80% train, 20% temp
X_train, y_train, X_temp, y_temp = iterative_train_test_split(
    X, y, test_size=0.2
)

# Second split: 50% of temp = val, 50% of temp = test (10% each overall)
X_val, y_val, X_test, y_test = iterative_train_test_split(
    X_temp, y_temp, test_size=0.5
)

print(f"Train : {len(X_train)} images")
print(f"Val   : {len(X_val)} images")
print(f"Test  : {len(X_test)} images")
print(f"Total : {len(X_train)+len(X_val)+len(X_test)} images")

Train : 12000 images
Val   : 1500 images
Test  : 1500 images
Total : 15000 images


In [ ]:
def make_split_df(X, y, label_cols, label_df):
    ids = [x[0] for x in X]
    rows = []
    for i, img_id in enumerate(ids):
        row = {'image_id': img_id}
        for j, col in enumerate(label_cols):
            row[col] = y[i][j]
        rows.append(row)
    return pd.DataFrame(rows)

train_split = make_split_df(X_train, y_train, label_cols, label_df)
val_split   = make_split_df(X_val,   y_val,   label_cols, label_df)
test_split  = make_split_df(X_test,  y_test,  label_cols, label_df)

# Separate test into pathological and healthy
test_patho   = test_split[test_split[label_cols].sum(axis=1) > 0]
test_healthy = test_split[test_split[label_cols].sum(axis=1) == 0]

# Save splits
train_split.to_csv(str(SPLITS_OUT / "train.csv"),        index=False)
val_split.to_csv(str(SPLITS_OUT   / "val.csv"),          index=False)
test_split.to_csv(str(SPLITS_OUT  / "test.csv"),         index=False)
test_patho.to_csv(str(SPLITS_OUT  / "test_patho.csv"),   index=False)
test_healthy.to_csv(str(SPLITS_OUT/ "test_healthy.csv"), index=False)

print("✅ Splits saved:")
print(f"   train.csv        : {len(train_split)} images")
print(f"   val.csv          : {len(val_split)} images")
print(f"   test.csv         : {len(test_split)} images")
print(f"   test_patho.csv   : {len(test_patho)} images")
print(f"   test_healthy.csv : {len(test_healthy)} images")

✅ Splits saved:
   train.csv        : 12000 images
   val.csv          : 1500 images
   test.csv         : 1500 images
   test_patho.csv   : 461 images
   test_healthy.csv : 1039 images


## Step 7 — Final verification


In [ ]:
print("── NB01 Verification ───────────────────────────────────")

processed_imgs  = sum(1 for _ in IMAGES_OUT.glob("*.png"))
consensus_files = sum(1 for _ in CONSENSUS_OUT.glob("*.csv"))
split_files     = sum(1 for _ in SPLITS_OUT.glob("*.csv"))
kappa_exists    = (PROCESSED_PATH / "fleiss_kappa.csv").exists()

print(f"224×224 PNGs in processed/images/  : {processed_imgs}  (expected ~15,000)")
print(f"Consensus CSV files                : {consensus_files}  (expected: 2)")
print(f"Split CSV files                    : {split_files}  (expected: 5)")
print(f"fleiss_kappa.csv exists            : {kappa_exists}")
print()

all_ok = (
    processed_imgs > 14000 and
    consensus_files == 2 and
    split_files == 5 and
    kappa_exists
)

if all_ok:
    print("✅ NB01 complete. Ready to run NB02_model_training.ipynb")
else:
    print("⚠️  Something is missing — check outputs above.")

── NB01 Verification ───────────────────────────────────
224×224 PNGs in processed/images/  : 15000  (expected ~15,000)
Consensus CSV files                : 2  (expected: 2)
Split CSV files                    : 5  (expected: 5)
fleiss_kappa.csv exists            : True

✅ NB01 complete. Ready to run NB02_model_training.ipynb


## ✅ NB01 Complete

| Step | Output | Status |
|---|---|---|
| Resize to 224×224 | `data/processed/images/` | ✅ |
| Consensus boxes | `data/processed/consensus/` | ✅ |
| Fleiss' κ | `data/processed/fleiss_kappa.csv` | ✅ |
| Train/Val/Test splits | `data/processed/splits/` | ✅ |

**Next step → Open `notebooks/NB02_model_training.ipynb`**